# 🔍 Vietnamese Fake News Detection System - Colab Test Notebook

Notebook này được thiết kế để chạy trực tiếp trên **Google Colab**. Nó sẽ tự động tải mã nguồn từ GitHub, cài đặt môi trường và chạy thử nghiệm hệ thống.

**Tác giả:** UET Project Team

## 📥 Bước 1: Clone Repository từ GitHub

Tải toàn bộ source code của dự án về môi trường Colab và di chuyển vào thư mục dự án.

In [ ]:
# Xóa thư mục cũ nếu đã clone trước đó để tránh lỗi
!rm -rf fake-news-detection

# Clone repository (thay link bằng link thực tế nếu cần)
!git clone https://github.com/phidinhmanh/fake-news-detection.git

# Di chuyển thư mục làm việc vào trong project
%cd fake-news-detection

## 🛠️ Bước 2: Cài đặt Dependencies

Cài đặt các thư viện xử lý tiếng Việt (`underthesea`), NLP (`transformers`), Vector Database (`lancedb`), và các công cụ khác.

In [ ]:
!pip install -q underthesea transformers datasets lancedb sentence-transformers google-generativeai langdetect wordcloud plotly

## ⚙️ Bước 3: Thiết lập môi trường Python

Đảm bảo Python nhận diện đúng thư mục gốc của project để có thể import các module (`config`, `agents`, `dataset`,...).

In [ ]:
import os
import sys
from pathlib import Path

ROOT_DIR = Path(".").resolve()
sys.path.append(str(ROOT_DIR))
print(f"Project Root: {ROOT_DIR}")

## 📂 Bước 4: Chuẩn bị dữ liệu (Mock data)

Tạo dữ liệu giả lập để test ngay pipeline mà không cần phải tải dataset thật về.

In [ ]:
import config
import pandas as pd

config.DATASET_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
config.KB_DIR.mkdir(parents=True, exist_ok=True)

mock_data = pd.DataFrame({
    'text': [
        'Vaccine COVID-19 cực kỳ nguy hiểm, hãy chia sẻ ngay!',
        'Thủ tướng họp về phát triển kinh tế xã hội quý 1.',
        'Giá xăng tăng kỷ lục vào ngày mai.',
    ],
    'label': ['fake', 'real', 'fake'],
    'source': ['social', 'news', 'news'],
    'lang': ['vi', 'vi', 'vi']
})
mock_path = config.DATASET_PROCESSED_DIR / "preprocessed_all.csv"
mock_data.to_csv(mock_path, index=False)
print(f"Created mock data at {mock_path}")

## 📊 Bước 5: Chạy Pipeline Xử lý dữ liệu

Test chức năng gộp dữ liệu và trích xuất đặc trưng phong cách.

In [ ]:
!python dataset/merge_datasets.py
!python dataset/feature_extraction.py

## 🤖 Bước 6: Test LLM Agent Pipeline (Mock Mode)

Test bộ 3 Agents ở chế độ Mock (không gọi thật lên API) để xem pipeline có nối với nhau mượt mà không.

In [ ]:
from agents.agent_pipeline import AgentPipeline

# Chạy chế độ mock để không cần cấu hình API Key
pipeline = AgentPipeline(mock=True, use_wikipedia=False)

test_article = """
KHẨN CẤP: Vaccine COVID-19 gây ra hàng nghìn ca tử vong trên toàn thế giới! 
Theo một nghiên cứu bí mật bị rò rỉ, các hãng dược phẩm đã che giấu sự thật 
về tác dụng phụ nghiêm trọng của vaccine. Hãy chia sẻ thông tin này ngay!!!
"""

print("🚀 Đang phân tích bài viết...")
result = pipeline.analyze(test_article)

print("\n--- KẾT QUẢ ---")
print(f"Label: {result.label}")
print(f"Điểm tin giả (Fake Score): {result.fake_score}%")
print(f"Giải thích: {result.explanation}")
print(f"Số claims trích xuất được: {len(result.claims)}")

## 🧠 Bước 7: Thử nghiệm Model PhoBERT

Tải cấu trúc PhoBERT và tokenizer. (Sẽ tải pre-trained weights từ HuggingFace nên cần Internet).

In [ ]:
import torch
from transformers import AutoTokenizer
from model.phobert_baseline import PhoBERTBaseline

try:
    tokenizer = PhoBERTBaseline.get_tokenizer()
    print("✅ Tải PhoBERT Tokenizer thành công!")
    
    # Thử tokenize 1 câu tiếng Việt
    inputs = tokenizer("Tôi là sinh viên đại học Công Nghệ", return_tensors="pt")
    print(f"Tokens: {tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])}")
except Exception as e:
    print(f"❌ Lỗi: {e}")

## 📋 Bước 8: Tiếp theo...

Hệ thống đã hoạt động trên Colab!

**Các bước tự thực hiện tiếp:**
1. Đổi `mock=False` và thêm `GOOGLE_API_KEY` vào `.env` để chạy Agent với API thật.
2. Tải ViFactCheck thật bằng cách dùng file `download_datasets.py`.
3. Có thể bắt đầu train `phobert_baseline.py` bằng Colab GPU.